In [ ]:
import json
import os
import glob
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Make the project root importable so `from src.bots import ...` works
sys.path.insert(0, str(Path.cwd().parent))

## Load data files & Mouse trajectory preview

In [ ]:
from src.plotting import plot_trajectory
from src.data import find_red_eclipse_files, load_red_eclipse_mouse

game_files = find_red_eclipse_files()

for file_path in game_files[:5]:
    meta, mouse = load_red_eclipse_mouse(file_path)
    if mouse.empty:
        continue
    plot_trajectory(mouse, title=f"userId={meta['userId']}, gameId={meta['gameId']}")
    print(f"\nuserId={meta['userId']}, gameId={meta['gameId']}, events={len(mouse)}")


## Feature extraction


In [ ]:
from src.features import extract_features
from src.data import load_red_eclipse_mouse

game_rows = []
for file_path in game_files:
    meta, mouse = load_red_eclipse_mouse(file_path)
    features = extract_features(mouse)

    if features is None:
        continue

    features.update({
        **meta,
        "is_bot": 0,
        "bot_type": "human",
    })
    game_rows.append(features)

games_df = pd.DataFrame(game_rows)
games_df.to_csv("../data/red_eclipse_features.csv", index=False)
print(games_df.head())


## Player identification


In [ ]:
from src.features import feature_cols

MIN_GAMES = 8
games_df_37 = games_df.groupby("userId").filter(lambda g: len(g) >= MIN_GAMES)

# games_df = all 45 players, games_df_37 = more than 8 games
select_model = games_df_37

input_data = select_model[feature_cols]
output_data = select_model["userId"]

input_train, input_test, output_train, output_test = train_test_split(
    input_data, output_data, test_size=0.2, random_state=42, stratify=output_data
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(input_train, output_train)

output_pred = model.predict(input_test)
accuracy = accuracy_score(output_test, output_pred)

print(f"Accuracy: {accuracy:.2%}")
print(f"Random baseline: {1 / output_data.nunique():.2%} ({output_data.nunique()} players, {len(select_model)} games)")
print()
print(classification_report(output_test, output_pred))


## Block-bootstrap synthetic bots


In [ ]:
from src.bots import build_segments, stitch_bot_game
from src.features import extract_features
from src.config import RNG_SEED
from src.data import load_red_eclipse_mouse

N_BOT_GAMES = len(games_df)  # one bot game per human game

rng = np.random.default_rng(RNG_SEED)

# 1) Build segment pool from human games
segment_pool = []
for file_path in game_files:
    meta, mouse = load_red_eclipse_mouse(file_path)
    segment_pool.extend(build_segments(mouse))

print(f"Segment pool: {len(segment_pool)} segments from {len(game_files)} games")

# 2) Generate synthetic bot games
N_PREVIEW = 5              # save first N trajectories for render graph
bot_rows = []
sample_bot_trajectories = []
for i in range(N_BOT_GAMES):
    bot_mouse = stitch_bot_game(segment_pool, rng=rng)
    if len(sample_bot_trajectories) < N_PREVIEW:
        sample_bot_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -1,
        "gameId": f"bot_{i}",
        "source_file": f"synthetic_bot_{i}",
        "is_bot": 1,
        "bot_type": "stitch",
    })
    bot_rows.append(feats)

bots_stitch_df = pd.DataFrame(bot_rows)
print(f"Generated {len(bots_stitch_df)} stitch bot games")
print(bots_stitch_df.head())


## Block-bootstrap synthetic bots trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_bot_trajectories):
    plot_trajectory(df, title=f"bot_{i}")
    print(f"\nbot_{i}, events={len(df)}")


## Smooth bot generation


In [ ]:
from src.bots import estimate_smooth_params, generate_smooth_bot_game
from src.features import extract_features

median_events = int(games_df["n_events"].median())
N_SMOOTH_BOTS = len(games_df)

re_smooth_params = estimate_smooth_params(games_df)
print(f"RE smooth params: {re_smooth_params}")

smooth_rows = []
sample_smooth_trajectories = []
for i in range(N_SMOOTH_BOTS):
    bot_mouse = generate_smooth_bot_game(n_events=median_events, seed=RNG_SEED + i, **re_smooth_params)
    if len(sample_smooth_trajectories) < N_PREVIEW:
        sample_smooth_trajectories.append(bot_mouse.copy())

    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -2,
        "gameId": f"smooth_{i}",
        "source_file": f"synthetic_smooth_{i}",
        "is_bot": 1,
        "bot_type": "smooth",
    })
    smooth_rows.append(feats)

bots_smooth_df = pd.DataFrame(smooth_rows)
print(f"Generated {len(bots_smooth_df)} smooth bot games (n_events={median_events})")
print(bots_smooth_df.head())


## Smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_smooth_trajectories):
    plot_trajectory(df, title=f"smooth_{i}")
    print(f"\nsmooth_{i}, events={len(df)}")

## RE Human vs Bot classification


In [ ]:
from src.evaluation import train_bot_detector
from src.features import feature_cols

re_model_stitch, re_acc_stitch = train_bot_detector(
    games_df, bots_stitch_df, feature_cols, name="stitch"
)
re_model_smooth, re_acc_smooth = train_bot_detector(
    games_df, bots_smooth_df, feature_cols, name="smooth"
)

print("RE in-domain bot detection:")
print(f"  stitch: {re_acc_stitch:.2%}")
print(f"  smooth: {re_acc_smooth:.2%}")


## Load LoL dataset

In [ ]:
from src.data import parse_lol_keylogger, find_lol_keylogger_files

RE_TRAIN_N = None
LOL_FILE_N = None
LOL_WINDOW_MIN = 3
LOL_PARSE_MAX_EVENTS = None
N_LOL_BOTS = None
SKIP_FULL_LOL_TRAJECTORY = False

lol_keylogger_files = find_lol_keylogger_files()

sample_lol = parse_lol_keylogger(
    lol_keylogger_files[0],
    max_events=LOL_PARSE_MAX_EVENTS,
    max_minutes=LOL_WINDOW_MIN,
)
print(f"File: {lol_keylogger_files[0].name}")
print(f"Events: {len(sample_lol)}, duration: {sample_lol['time'].iloc[-1] / 60000:.1f} min")
print(sample_lol.head())


## Load LoL sessions & extract features

In [ ]:
from src.features import extract_features
from src.data import parse_lol_keylogger, find_lol_keylogger_files

lol_keylogger_files = find_lol_keylogger_files()

files_to_load = lol_keylogger_files[:LOL_FILE_N] if LOL_FILE_N else lol_keylogger_files
print(f"Loading {len(files_to_load)} LoL files")

lol_rows = []
for file_path in files_to_load:
    mouse = parse_lol_keylogger(
        file_path,
        max_events=LOL_PARSE_MAX_EVENTS,
        max_minutes=LOL_WINDOW_MIN,
    )
    if mouse is None:
        continue
    mouse = mouse[mouse["time"] <= LOL_WINDOW_MIN * 60 * 1000]
    feats = extract_features(mouse)
    if feats is None:
        continue

    session_date = file_path.parent.name
    participant = file_path.stem.split("-")[1]
    feats.update({
        "userId": f"lol_{participant}",
        "gameId": f"{session_date}_p{participant}",
        "source_file": file_path.name,
        "session_date": session_date,
        "is_bot": 0,
        "bot_type": "human",
    })
    lol_rows.append(feats)

lol_games_df = pd.DataFrame(lol_rows)
print(f"Loaded {len(lol_games_df)} LoL human sessions (first {LOL_WINDOW_MIN} min each)")
print(lol_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
print(f"Red Eclipse — median n_events: {games_df['n_events'].median():.0f}")
print(f"LoL — median n_events: {lol_games_df['n_events'].median():.0f}")


## LoL trajectory preview

In [ ]:
from src.data import parse_lol_keylogger, find_lol_keylogger_files

lol_keylogger_files = find_lol_keylogger_files()

PREVIEW_MINUTES = LOL_WINDOW_MIN

preview_lol = parse_lol_keylogger(
    lol_keylogger_files[0],
    max_events=LOL_PARSE_MAX_EVENTS,
    max_minutes=PREVIEW_MINUTES,
)
preview_lol = preview_lol[preview_lol["time"] <= PREVIEW_MINUTES * 60 * 1000].copy()
preview_lol["trajectory_x"] = preview_lol["dx"].cumsum()
preview_lol["trajectory_y"] = preview_lol["dy"].cumsum()

print(f"Zoom: first {PREVIEW_MINUTES} min, {len(preview_lol)} events")

if SKIP_FULL_LOL_TRAJECTORY:
    plt.figure(figsize=(7, 5))
    plt.plot(preview_lol["trajectory_x"], preview_lol["trajectory_y"], linewidth=0.5, color="steelblue")
    plt.title(f"LoL — first {PREVIEW_MINUTES} min")
    plt.gca().invert_yaxis()
    plt.axis("equal")
    plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(preview_lol["trajectory_x"], preview_lol["trajectory_y"], linewidth=0.5, color="steelblue")
    axes[0].set_title(f"LoL — first {PREVIEW_MINUTES} min")
    axes[0].invert_yaxis()
    axes[0].set_aspect("equal")
    full_lol = parse_lol_keylogger(lol_keylogger_files[0])
    full_lol["trajectory_x"] = full_lol["dx"].cumsum()
    full_lol["trajectory_y"] = full_lol["dy"].cumsum()
    axes[1].plot(full_lol["trajectory_x"], full_lol["trajectory_y"], linewidth=0.1, color="steelblue", alpha=0.3)
    axes[1].set_title(f"LoL — full session ({len(full_lol)/1e6:.2f}M events)")
    axes[1].invert_yaxis()
    axes[1].set_aspect("equal")
    plt.tight_layout()
    plt.show()


## Cross-game transfer (RE train → LoL test)

In [ ]:
from src.features import cross_game_feature_cols
from src.evaluation import train_bot_detector

re_human = games_df.head(RE_TRAIN_N) if RE_TRAIN_N else games_df
re_stitch = bots_stitch_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_stitch_df
re_smooth = bots_smooth_df.head(RE_TRAIN_N) if RE_TRAIN_N else bots_smooth_df

print("=== RE model trained on STITCH bots ===")
re_model_stitch, re_acc_stitch = train_bot_detector(re_human, re_stitch, cross_game_feature_cols, name="stitch")
print()
print("=== RE model trained on SMOOTH bots ===")
re_model_smooth, re_acc_smooth = train_bot_detector(re_human, re_smooth, cross_game_feature_cols, name="smooth")


## LoL data human identifier

In [ ]:
# test will LoL humans identified as bot
for name, model in [("stitch", re_model_stitch), ("smooth", re_model_smooth)]:
    pred = model.predict(lol_games_df[cross_game_feature_cols])
    print(f"[{name} model] LoL human false-positive rate: {pred.mean():.2%} ({pred.sum()}/{len(pred)} identified as bot)")

## LoL stitch bot generation

In [ ]:
from src.bots import build_segments, stitch_bot_game
from src.data import parse_lol_keylogger

lol_segment_pool = []
for file_path in files_to_load:
    mouse = parse_lol_keylogger(
        file_path,
        max_events=LOL_PARSE_MAX_EVENTS,
        max_minutes=LOL_WINDOW_MIN,
    )
    if mouse is None:
        continue
    mouse = mouse[mouse["time"] <= LOL_WINDOW_MIN * 60 * 1000]
    lol_segment_pool.extend(build_segments(mouse))

n_lol_bots = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)
target_ms = LOL_WINDOW_MIN * 60 * 1000
lol_stitch_rows = []
sample_lol_stitch_trajectories = []
lol_bot_rng = np.random.default_rng(RNG_SEED + 1)

for i in range(n_lol_bots):
    bot_mouse = stitch_bot_game(lol_segment_pool, target_duration_ms=target_ms, rng=lol_bot_rng)
    if len(sample_lol_stitch_trajectories) < N_PREVIEW:
        sample_lol_stitch_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -1, "gameId": f"lol_stitch_{i}", "is_bot": 1, "bot_type": "stitch"})
    lol_stitch_rows.append(feats)

lol_bots_stitch_df = pd.DataFrame(lol_stitch_rows)
print(f"LoL stitch bots: {len(lol_bots_stitch_df)} (target {target_ms/1000:.0f}s each)")

## LoL stitch bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_stitch_trajectories):
    print(f"lol_stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"lol_stitch_{i}")

## LoL smooth bot generation

In [ ]:
from src.bots import estimate_smooth_params, generate_smooth_bot_game

lol_median_events = int(lol_games_df["n_events"].median())
n_lol_smooth = N_LOL_BOTS if N_LOL_BOTS else len(lol_games_df)

lol_smooth_params = estimate_smooth_params(lol_games_df)
print(f"LoL smooth params: {lol_smooth_params}")
print(f"(RE smooth params for comparison: {re_smooth_params})")

lol_smooth_rows = []
sample_lol_smooth_trajectories = []
for i in range(n_lol_smooth):
    bot_mouse = generate_smooth_bot_game(n_events=lol_median_events, seed=RNG_SEED + 100 + i, **lol_smooth_params)
    if len(sample_lol_smooth_trajectories) < N_PREVIEW:
        sample_lol_smooth_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({"userId": -2, "gameId": f"lol_smooth_{i}", "is_bot": 1, "bot_type": "smooth"})
    lol_smooth_rows.append(feats)

lol_bots_smooth_df = pd.DataFrame(lol_smooth_rows)
print(f"LoL smooth bots: {len(lol_bots_smooth_df)} (n_events={lol_median_events})")

## LoL smooth bot trajectory preview

In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_lol_smooth_trajectories):
    plot_trajectory(df, title=f"lol_smooth_{i}")
    print(f"lol_smooth_{i}, events={len(df)}")


## LoL in-domain sanity check

In [ ]:
from src.evaluation import train_bot_detector
from src.features import feature_cols

_, lol_re_acc_stitch = train_bot_detector(
    lol_games_df, lol_bots_stitch_df, feature_cols, name="LoL stitch"
)
print()
_, lol_re_acc_smooth = train_bot_detector(
    lol_games_df, lol_bots_smooth_df, feature_cols, name="LoL smooth"
)

print()
print("LoL in-domain bot detection (train+test on LoL):")
print(f"  stitch: {lol_re_acc_stitch:.2%}")
print(f"  smooth: {lol_re_acc_smooth:.2%}")


## Feature scale comparison (RE vs LoL)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

# --- Check 2: RE vs LoL 特徵尺度差多少？ ---
print("Feature medians (RE human vs LoL human vs LoL stitch bot):")
compare = pd.DataFrame({
    "RE_human": games_df[cols].median(),
    "LoL_human": lol_games_df[cols].median(),
    "LoL_stitch": lol_bots_stitch_df[cols].median(),
}).round(3)
print(compare)

## Zero-shot diagnose (raw features, RE → LoL)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, lol_games_df, lol_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, lol_games_df, lol_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Scale-free train (RE)

Train RF on unitless ratio features (same RE subset as raw cross-game models).


In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_free, SCALE_FREE_COLS
from src.config import RNG_SEED

# Same RE training subset as raw cross-game models when RE_TRAIN_N is set
re_sf_human = to_scale_free(re_human)
re_sf_stitch = to_scale_free(re_stitch)
re_sf_smooth = to_scale_free(re_smooth)

print("=== Train on Red Eclipse (scale-free features) ===")
m_sf_stitch, _ = train_bot_detector(
    re_sf_human, re_sf_stitch, SCALE_FREE_COLS,
    random_state=RNG_SEED, name="scale-free stitch",
    show_feature_importance=False,
)
print()
m_sf_smooth, _ = train_bot_detector(
    re_sf_human, re_sf_smooth, SCALE_FREE_COLS,
    random_state=RNG_SEED, name="scale-free smooth",
    show_feature_importance=False,
)


## Zero-shot diagnose (scale-free features, RE → LoL)

Same `diagnose_zero_shot` suite as raw: AUC, P(bot), @0.5, human-calibrated threshold, sweep.


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_free, SCALE_FREE_COLS

lol_sf_human = to_scale_free(lol_games_df)
lol_sf_stitch = to_scale_free(lol_bots_stitch_df)
lol_sf_smooth = to_scale_free(lol_bots_smooth_df)

print("=== Scale-free: true zero-shot RE -> LoL ===")
print("(threshold from LoL humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_sf_stitch, lol_sf_human, lol_sf_stitch,
    SCALE_FREE_COLS, title_suffix="scale-free",
)
print()
_ = diagnose_cross_game(
    "smooth", m_sf_smooth, lol_sf_human, lol_sf_smooth,
    SCALE_FREE_COLS, title_suffix="scale-free",
)


## Feature importance (scale-free models)


In [ ]:
import pandas as pd
from src.features import SCALE_FREE_COLS

imp = pd.Series(m_sf_stitch.feature_importances_, index=SCALE_FREE_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-free stitch model) ===")
print(imp)
print()
imp2 = pd.Series(m_sf_smooth.feature_importances_, index=SCALE_FREE_COLS).sort_values(ascending=False)
print("=== Feature importance (scale-free smooth model) ===")
print(imp2)


## CSGO load: eye_vector → (dx, dy, time)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from src.config import CSGO_DATA_ROOT
from src.data_csgo import (
    check_axis_convention,
    eye_vectors_to_mouse_df,
    validate_real_roundtrip,
)

USECOLS = ["time", "eyeVectorX", "eyeVectorY", "eyeVectorZ"]

def find_gameflt_files(root=CSGO_DATA_ROOT):
    files = sorted(Path(root).rglob("gameFlt.csv"))
    print(f"Found {len(files)} gameFlt.csv under {root}")
    return files

def load_eye_csv(path):
    df = pd.read_csv(path, usecols=USECOLS)
    finite = np.isfinite(df[["eyeVectorX", "eyeVectorY", "eyeVectorZ"]]).all(axis=1)
    return df.loc[finite].reset_index(drop=True)

def convert_one(path):
    """One file → mouse DataFrame(dx, dy, time_ms) + meta."""
    flt = load_eye_csv(path)
    mouse_df, meta = eye_vectors_to_mouse_df(
        flt["time"], flt["eyeVectorX"], flt["eyeVectorY"], flt["eyeVectorZ"]
    )
    return mouse_df, meta, flt

gameflt_paths = find_gameflt_files()
assert gameflt_paths, f"No gameFlt.csv under {CSGO_DATA_ROOT}"

first_path = gameflt_paths[0]
print(f"\n=== First-file checks: {first_path} ===")

mouse0, meta0, flt0 = convert_one(first_path)
print("convert meta:", meta0)
print(mouse0.head())

axis_ok = check_axis_convention(flt0["eyeVectorY"])
rt = validate_real_roundtrip(
    flt0["eyeVectorX"], flt0["eyeVectorY"], flt0["eyeVectorZ"]
)

if not (axis_ok and rt["ok"]):
    raise RuntimeError(
        "First-file checks failed — stop before converting all files. "
        f"axis_ok={axis_ok}, roundtrip_ok={rt['ok']}"
    )

print("\nFirst file PASSED. Converting all gameFlt.csv ...")

csgo_mouse = {}
rows = []
for i, path in enumerate(gameflt_paths, 1):
    # path like .../S001/P3/gameFlt.csv
    participant = path.parent.name          # P3
    session = path.parent.parent.name       # S001
    key = (session, participant)
    try:
        mouse_df, meta, _ = convert_one(path)
    except Exception as e:
        print(f"  SKIP {key}: {e}")
        continue
    csgo_mouse[key] = mouse_df
    rows.append({
        "session": session,
        "participant": participant,
        "n_out": meta["n_out"],
        "teleport_frac": meta["teleport_frac"],
        "path": str(path),
    })
    if i % 50 == 0 or i == len(gameflt_paths):
        print(f"  converted {i}/{len(gameflt_paths)}")

csgo_convert_summary = pd.DataFrame(rows)
print(f"\nDone: {len(csgo_mouse)} traces")
print(csgo_convert_summary.head())
print(
    "n_out median:", csgo_convert_summary["n_out"].median(),
    "| teleport_frac median:", f"{csgo_convert_summary['teleport_frac'].median():.2%}",
)

## CSGO sessions & extract features


In [ ]:
from pathlib import Path

from src.features import extract_features
from src.config import CSGO_DATA_ROOT, CSGO_WINDOW_MIN
from src.data_csgo import window_mouse_round_alive

print(
    f"Extracting features from {len(csgo_mouse)} CSGO traces "
    f"(Round2+alive, {CSGO_WINDOW_MIN} min)"
)

csgo_mouse_win = {}
csgo_rows = []
csgo_window_meta = []
n_skip = 0

for (session, participant), mouse in csgo_mouse.items():
    session_dir = Path(CSGO_DATA_ROOT) / session / participant
    win, meta = window_mouse_round_alive(
        mouse, session_dir, window_min=CSGO_WINDOW_MIN
    )
    csgo_window_meta.append({"session": session, "participant": participant, **meta})
    if not meta.get("ok") or win is None:
        n_skip += 1
        continue

    feats = extract_features(win)
    if feats is None:
        n_skip += 1
        continue

    csgo_mouse_win[(session, participant)] = win
    feats.update({
        "userId": f"csgo_{session}_{participant}",
        "gameId": f"{session}_{participant}",
        "session": session,
        "participant": participant,
        "is_bot": 0,
        "bot_type": "human",
        "round_n": meta["round_n"],
        "alive_frac_in_window": meta["alive_frac_in_window"],
    })
    csgo_rows.append(feats)

csgo_games_df = pd.DataFrame(csgo_rows)
csgo_window_meta_df = pd.DataFrame(csgo_window_meta)

print(
    f"Loaded {len(csgo_games_df)} CSGO human sessions "
    f"(Round2+alive, skipped {n_skip})"
)
print(
    "alive_frac_in_window median:",
    f"{csgo_games_df['alive_frac_in_window'].median():.1%}",
)
print(csgo_games_df[["n_events", "total_movement", "avg_speed", "idle_ratio"]].describe())
print()
print(f"Red Eclipse — median n_events: {games_df['n_events'].median():.0f}")
print(f"LoL — median n_events: {lol_games_df['n_events'].median():.0f}")
print(f"CSGO — median n_events: {csgo_games_df['n_events'].median():.0f}")
print(csgo_games_df.head())


## CSGO trajectory preview


In [ ]:
from src.plotting import plot_trajectory
from src.config import CSGO_WINDOW_MIN

preview_keys = list(csgo_mouse_win.keys())[:5]

for session, participant in preview_keys:
    mouse = csgo_mouse_win[(session, participant)]
    plot_trajectory(
        mouse,
        title=f"CSGO {session}/{participant} (Round2+alive, {CSGO_WINDOW_MIN} min)",
    )
    print(f"\n{session}/{participant}, events={len(mouse)}")


## CSGO stitch bot generation


In [ ]:
from src.bots import build_segments, stitch_bot_game
from src.features import extract_features
from src.config import RNG_SEED, CSGO_WINDOW_MIN

N_PREVIEW = 5
N_CSGO_BOTS = None  # None = one bot per CSGO human session

csgo_segment_pool = []
for mouse in csgo_mouse_win.values():
    csgo_segment_pool.extend(build_segments(mouse))

print(
    f"CSGO segment pool: {len(csgo_segment_pool)} segments "
    f"from {len(csgo_mouse_win)} Round2+alive traces"
)

n_csgo_bots = N_CSGO_BOTS if N_CSGO_BOTS else len(csgo_games_df)
target_ms = CSGO_WINDOW_MIN * 60 * 1000
csgo_stitch_rows = []
sample_csgo_stitch_trajectories = []
csgo_bot_rng = np.random.default_rng(RNG_SEED + 2)

for i in range(n_csgo_bots):
    bot_mouse = stitch_bot_game(
        csgo_segment_pool, target_duration_ms=target_ms, rng=csgo_bot_rng
    )
    if len(sample_csgo_stitch_trajectories) < N_PREVIEW:
        sample_csgo_stitch_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -1,
        "gameId": f"csgo_stitch_{i}",
        "is_bot": 1,
        "bot_type": "stitch",
    })
    csgo_stitch_rows.append(feats)

csgo_bots_stitch_df = pd.DataFrame(csgo_stitch_rows)
print(f"CSGO stitch bots: {len(csgo_bots_stitch_df)} (target {target_ms/1000:.0f}s each)")
print(csgo_bots_stitch_df.head())


## CSGO stitch bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_stitch_trajectories):
    print(f"csgo_stitch_{i}, events={len(df)}")
    plot_trajectory(df, title=f"csgo_stitch_{i}")


## CSGO smooth bot generation


In [ ]:
from src.bots import estimate_smooth_params, generate_smooth_bot_game
from src.features import extract_features
from src.config import RNG_SEED

csgo_median_events = int(csgo_games_df["n_events"].median())
n_csgo_smooth = N_CSGO_BOTS if N_CSGO_BOTS else len(csgo_games_df)

csgo_smooth_params = estimate_smooth_params(csgo_games_df)
print(f"CSGO smooth params: {csgo_smooth_params}")

csgo_smooth_rows = []
sample_csgo_smooth_trajectories = []
for i in range(n_csgo_smooth):
    # round_deltas=False: CSGO dx/dy are degrees (often << 1); integer round would wipe them
    bot_mouse = generate_smooth_bot_game(
        n_events=csgo_median_events,
        seed=RNG_SEED + 200 + i,
        round_deltas=False,
        **csgo_smooth_params,
    )
    if len(sample_csgo_smooth_trajectories) < N_PREVIEW:
        sample_csgo_smooth_trajectories.append(bot_mouse.copy())
    feats = extract_features(bot_mouse)
    if feats is None:
        continue
    feats.update({
        "userId": -2,
        "gameId": f"csgo_smooth_{i}",
        "is_bot": 1,
        "bot_type": "smooth",
    })
    csgo_smooth_rows.append(feats)

csgo_bots_smooth_df = pd.DataFrame(csgo_smooth_rows)
print(f"CSGO smooth bots: {len(csgo_bots_smooth_df)} (n_events={csgo_median_events})")
print(csgo_bots_smooth_df.head())


## CSGO smooth bot trajectory preview


In [ ]:
from src.plotting import plot_trajectory

for i, df in enumerate(sample_csgo_smooth_trajectories):
    plot_trajectory(df, title=f"csgo_smooth_{i}")
    print(f"csgo_smooth_{i}, events={len(df)}")


## CSGO in-domain sanity check

Uses Round2+alive human windows and bots built from the same cut.


In [ ]:
from src.evaluation import train_bot_detector
from src.features import feature_cols

csgo_model_stitch, csgo_acc_stitch = train_bot_detector(
    csgo_games_df, csgo_bots_stitch_df, feature_cols, name="CSGO stitch"
)
print()
csgo_model_smooth, csgo_acc_smooth = train_bot_detector(
    csgo_games_df, csgo_bots_smooth_df, feature_cols, name="CSGO smooth"
)

print()
print("CSGO in-domain bot detection (train+test on CSGO):")
print(f"  stitch: {csgo_acc_stitch:.2%}")
print(f"  smooth: {csgo_acc_smooth:.2%}")


## Zero-shot diagnose (raw features, RE → CSGO)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", re_model_stitch, csgo_games_df, csgo_bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", re_model_smooth, csgo_games_df, csgo_bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Scale-free train (RE) — for CSGO transfer

In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_free, SCALE_FREE_COLS
from src.config import RNG_SEED

re_sf_human = to_scale_free(re_human)
re_sf_stitch = to_scale_free(re_stitch)
re_sf_smooth = to_scale_free(re_smooth)

print("=== Train on Red Eclipse (scale-free features) ===")
m_sf_stitch, _ = train_bot_detector(
    re_sf_human, re_sf_stitch, SCALE_FREE_COLS,
    random_state=RNG_SEED, name="scale-free stitch",
    show_feature_importance=False,
)
print()
m_sf_smooth, _ = train_bot_detector(
    re_sf_human, re_sf_smooth, SCALE_FREE_COLS,
    random_state=RNG_SEED, name="scale-free smooth",
    show_feature_importance=False,
)


## Zero-shot diagnose (scale-free features, RE → CSGO)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_free, SCALE_FREE_COLS

csgo_sf_human = to_scale_free(csgo_games_df)
csgo_sf_stitch = to_scale_free(csgo_bots_stitch_df)
csgo_sf_smooth = to_scale_free(csgo_bots_smooth_df)

print("=== Scale-free: true zero-shot RE -> CSGO ===")
print("(threshold from CSGO humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_sf_stitch, csgo_sf_human, csgo_sf_stitch,
    SCALE_FREE_COLS, title_suffix="scale-free",
)
print()
_ = diagnose_cross_game(
    "smooth", m_sf_smooth, csgo_sf_human, csgo_sf_smooth,
    SCALE_FREE_COLS, title_suffix="scale-free",
)


## Feature scale comparison (RE vs CSGO)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (RE human vs CSGO human vs CSGO bots):")
compare_re_csgo = pd.DataFrame({
    "RE_human": games_df[cols].median(),
    "CSGO_human": csgo_games_df[cols].median(),
    "CSGO_stitch": csgo_bots_stitch_df[cols].median(),
    "CSGO_smooth": csgo_bots_smooth_df[cols].median(),
}).round(3)
print(compare_re_csgo)


## Cross-game transfer (CSGO train → RE test)

In [ ]:
from src.features import cross_game_feature_cols
from src.evaluation import train_bot_detector

print("=== CSGO model trained on STITCH bots (7 cross-game features) ===")
csgo_x_model_stitch, csgo_x_acc_stitch = train_bot_detector(
    csgo_games_df, csgo_bots_stitch_df, cross_game_feature_cols, name="CSGO stitch"
)
print()
print("=== CSGO model trained on SMOOTH bots (7 cross-game features) ===")
csgo_x_model_smooth, csgo_x_acc_smooth = train_bot_detector(
    csgo_games_df, csgo_bots_smooth_df, cross_game_feature_cols, name="CSGO smooth"
)


## Zero-shot diagnose (raw features, CSGO → RE)

In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import cross_game_feature_cols

print("=== Raw features: true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", csgo_x_model_stitch, games_df, bots_stitch_df,
    cross_game_feature_cols, title_suffix="raw",
)
print()
_ = diagnose_cross_game(
    "smooth", csgo_x_model_smooth, games_df, bots_smooth_df,
    cross_game_feature_cols, title_suffix="raw",
)


## Scale-free train (CSGO) — for RE transfer


In [ ]:
from src.evaluation import train_bot_detector
from src.features import to_scale_free, SCALE_FREE_COLS
from src.config import RNG_SEED

csgo_sf_human_tr = to_scale_free(csgo_games_df)
csgo_sf_stitch_tr = to_scale_free(csgo_bots_stitch_df)
csgo_sf_smooth_tr = to_scale_free(csgo_bots_smooth_df)

print("=== Train on CSGO (scale-free features) ===")
m_csgo_sf_stitch, _ = train_bot_detector(
    csgo_sf_human_tr, csgo_sf_stitch_tr, SCALE_FREE_COLS,
    random_state=RNG_SEED, name="CSGO scale-free stitch",
    show_feature_importance=False,
)
print()
m_csgo_sf_smooth, _ = train_bot_detector(
    csgo_sf_human_tr, csgo_sf_smooth_tr, SCALE_FREE_COLS,
    random_state=RNG_SEED, name="CSGO scale-free smooth",
    show_feature_importance=False,
)


## Zero-shot diagnose (scale-free features, CSGO → RE)


In [ ]:
from src.evaluation import diagnose_cross_game
from src.features import to_scale_free, SCALE_FREE_COLS

re_sf_human = to_scale_free(games_df)
re_sf_stitch = to_scale_free(bots_stitch_df)
re_sf_smooth = to_scale_free(bots_smooth_df)

print("=== Scale-free: true zero-shot CSGO -> RE ===")
print("(threshold from RE humans only; bot labels used only for reporting)\n")

_ = diagnose_cross_game(
    "stitch", m_csgo_sf_stitch, re_sf_human, re_sf_stitch,
    SCALE_FREE_COLS, title_suffix="scale-free",
)
print()
_ = diagnose_cross_game(
    "smooth", m_csgo_sf_smooth, re_sf_human, re_sf_smooth,
    SCALE_FREE_COLS, title_suffix="scale-free",
)


## Feature scale comparison (CSGO vs RE)

In [ ]:
from src.features import cross_game_feature_cols

cols = cross_game_feature_cols

print("Feature medians (CSGO human vs RE human vs RE bots):")
compare_csgo_re = pd.DataFrame({
    "CSGO_human": csgo_games_df[cols].median(),
    "RE_human": games_df[cols].median(),
    "RE_stitch": bots_stitch_df[cols].median(),
    "RE_smooth": bots_smooth_df[cols].median(),
}).round(3)
print(compare_csgo_re)
